# CNN and CRNN training on five folds (S15a)

Trains the CNN (`time_pooling: mean`) and the CRNN (`time_pooling: lstm`) on all five official folds with the package's own command, `python -m acousticwise.train --model cnn`, on a GPU.

**Inputs:** the private `acousticwise-cache` dataset (the feature cache from S6) and the private `acousticwise-src` dataset (the package, `configs/default.yaml` and the split manifest, uploaded with `notebooks/spectrogram_grid/upload_src.sh`). GPU, no internet.

**Output** in `/kaggle/working/runs/`: one run directory per model and fold (`model.pt`, `config.json`, `labels.json`, `splits.json`, `metrics.json` with validation scores only), plus `checksums.txt`. The test split is not touched here; runs are evaluated locally with `python -m acousticwise.evaluate`.


## 1. Package, cache and manifest

In [ ]:
import glob
import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import torch
import yaml

# SHA-256 of the split manifest, from DATA.md.
MANIFEST_SHA256 = "84591f54320f34d6872f4b5de769869ba72eada9d8677fa98b37bac8e4916a18"

pkg = glob.glob("/kaggle/input/**/acousticwise/__init__.py", recursive=True)
assert pkg, "Attach the acousticwise-src dataset (Add Input)."
SRC = Path(pkg[0]).parent.parent
CONFIG = SRC / "configs" / "default.yaml"
MANIFEST = SRC / "artifacts" / "splits.json"
assert CONFIG.exists() and MANIFEST.exists(), "acousticwise-src lacks the config or manifest: rerun upload_src.sh."
assert hashlib.sha256(MANIFEST.read_bytes()).hexdigest() == MANIFEST_SHA256, "Manifest differs from DATA.md."

meta = glob.glob("/kaggle/input/**/sr32000/meta.json", recursive=True)
assert meta, "Attach the acousticwise-cache dataset (Add Input)."
CACHE = Path(meta[0]).parent.parent
RUNS = Path("/kaggle/working/runs")

print("code commit:", (SRC / "COMMIT").read_text().strip())
print("package:    ", SRC)
print("cache:      ", CACHE)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Turn on the GPU accelerator."


## 2. Config

The committed `configs/default.yaml`, with only the paths changed for Kaggle.

In [ ]:
cfg = yaml.safe_load(CONFIG.read_text())
cfg["data"]["cache_dir"] = str(CACHE)
cfg["data"]["splits_path"] = str(MANIFEST)
cfg["data"]["runs_dir"] = str(RUNS)
RUN_CONFIG = Path("/kaggle/working/config.yaml")
RUN_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(yaml.safe_dump({k: cfg[k] for k in ("seed", "audio", "cnn")}, sort_keys=False))


## 3. Train

Ten runs, one after another: the CNN on folds 1–5, then the CRNN. Each prints its best validation scores. A run already in `runs/` is skipped, so a restarted session continues.

In [ ]:
env = {**os.environ, "PYTHONPATH": str(SRC)}
for pooling, kind in [("mean", "cnn"), ("lstm", "crnn")]:
    for fold in range(1, 6):
        if list(RUNS.glob(f"{kind}_fold{fold}_*/metrics.json")):
            print(f"{kind} fold {fold}: already trained, skipped")
            continue
        t = time.time()
        result = subprocess.run(
            [sys.executable, "-m", "acousticwise.train", "--config", str(RUN_CONFIG),
             "--model", "cnn", "--fold", str(fold), "--time-pooling", pooling],
            env=env, capture_output=True, text=True,
        )
        print(result.stdout.strip() or result.stderr[-3000:])
        result.check_returncode()
        print(f"  ({time.time() - t:.0f} s wall)", flush=True)


## 4. Summary and checksums

In [ ]:
rows = []
for m in sorted(RUNS.glob("*/metrics.json")):
    v = json.loads(m.read_text())["validation"]
    rows.append((m.parent.name.split("_")[0], json.loads((m.parent / "config.json").read_text())["fold"],
                 v["macro_f1_headline"], v["macro_f1_all"], v["best_epoch"]))
print(f"{'model':6}{'fold':>5}{'val headline':>14}{'val all':>9}{'best epoch':>12}")
for kind, fold, head, all_, best in rows:
    print(f"{kind:6}{fold:5}{head:14.3f}{all_:9.3f}{best:12}")

with open(RUNS / "checksums.txt", "w") as f:
    for p in sorted(RUNS.rglob("*")):
        if p.is_file() and p.name != "checksums.txt":
            f.write(f"{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.relative_to(RUNS)}\n")
print((RUNS / "checksums.txt").read_text())
